# SRM INSTITUTE OF SCIENCE AND TECHNOLOGY, TIRUCHIRAPPALLI
## Department of Computer Science and Engineering
### 21CSE377T – Information Retrieval | CLA-2 Task 2
---
# TEXT CLASSIFICATION BATTLE: COMPARE, ANALYZE AND DEFEND
## News Category Classification Using Machine Learning

- **Student Name:** Sathish Kumar R
- **Register Number:** RA2411026050188
- **Institution:** SRM Institute of Science and Technology, Tiruchirappalli Campus
- **Course:** 21CSE377T – Information Retrieval
- **Assignment:** CLA-2 Task 2 – Text Classification Battle
- **Maximum Marks:** 30
- **Dataset:** BBC News Classification Corpus (2,225 articles across 5 categories)
- **Algorithms Evaluated:** Decision Tree, K-Nearest Neighbours, Support Vector Machine, Multinomial Naive Bayes

---
## 1. Problem Statement & Objectives
The automated organization and retrieval of unstructured textual content is a fundamental problem in Information Retrieval (IR). 
News agencies, digital aggregators, and search engines ingest tens of thousands of articles daily. Manual categorization is unscalable, prone to human fatigue, and slow.

**Project Objectives:**
1. Ingest and validate the authentic BBC News Classification corpus across 5 topical areas: *Business, Entertainment, Politics, Sport, Tech*.
2. Implement an end-to-end linguistic text preprocessing pipeline (lowercasing, punctuation/digit removal, tokenization, stop-word elimination, and WordNet lemmatization).
3. Transform textual documents into continuous high-dimensional vector spaces using **Term Frequency-Inverse Document Frequency (TF-IDF)** with strict data-leakage prevention.
4. Train, tune, and evaluate four fundamental machine learning algorithms:
   - **Decision Tree (CART)**
   - **K-Nearest Neighbours (KNN)**
   - **Support Vector Machine (Linear SVM)**
   - **Multinomial Naive Bayes (MNB)**
5. Benchmark performance using Accuracy, Precision, Recall, F1-Score (macro & weighted), 5-Fold Stratified Cross-Validation, and training/prediction latency.
6. Conduct parametric **What-If experiments** analyzing sensitivity to neighborhood size $k$, distance metric, vocabulary capacity, and n-gram ranges.


## 2. Environment Setup & Dependency Imports

In [ ]:
import os
import sys
import time
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure local source directory is accessible
sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing import TextPreprocessor
from src.model_training import load_and_preprocess_dataset, get_model_pipelines
from src.evaluation import evaluate_all_models, generate_evaluation_visualizations
from src.what_if_analysis import (
    run_knn_sensitivity_experiment,
    run_tfidf_feature_capacity_experiment,
    run_ngram_range_experiment,
    plot_what_if_experiments
)
from src.prediction import NewsPredictor

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Environment and dependencies initialized successfully.")


## 3. Dataset Ingestion & Exploratory Data Analysis (EDA)
The BBC News corpus consists of 2,225 genuine news articles published between 2004 and 2005.
We verify class balance, text length distributions, and duplicate records.


In [ ]:
df_raw = pd.read_csv('../dataset/bbc_news_clean.csv')
print(f"Total Records: {len(df_raw)}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"Missing Values:\n{df_raw.isnull().sum()}\n")
print("Class Distribution:")
print(df_raw['category'].value_counts())

df_raw['word_count'] = df_raw['text'].apply(lambda x: len(str(x).split()))
print("\nWord Count Statistics by Category:")
print(df_raw.groupby('category')['word_count'].agg(['count', 'mean', 'median', 'min', 'max']))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
counts = df_raw['category'].value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=ax1, palette='mako')
ax1.set_title("Article Frequency per Category", weight='bold')
ax1.set_ylabel("Document Count")
for p in ax1.patches:
    ax1.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width()/2., p.get_height()),
                 ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')

sns.boxplot(data=df_raw, x='category', y='word_count', ax=ax2, palette='Set2')
ax2.set_title("Word Count Distribution by Category", weight='bold')
ax2.set_ylim(0, 1500)
plt.tight_layout()
plt.show()


## 4. Text Preprocessing Pipeline & Demonstration
Raw text contains syntactic noise, capitalization inconsistencies, punctuation, stop words, and morphological inflections.
Our preprocessing pipeline applies:
1. **Lowercase Normalization**
2. **Regex URL / Tag Stripping**
3. **Punctuation and Numeric Elimination**
4. **Tokenization**
5. **Stop-Word Removal** (NLTK English lexicon)
6. **WordNet Lemmatization** (retaining morphological base forms)


In [ ]:
preprocessor = TextPreprocessor(use_stemmer=False, use_lemmatizer=True)
sample_text = "The UK Chancellor Gordon Brown announced a £1.5 billion investment in artificial intelligence and tech startups at 10 Downing Street on Friday!"

demo = preprocessor.demonstrate_steps(sample_text)
print("=== PREPROCESSING STEP-BY-STEP TRANSFORMATION ===")
print(f"1. Raw Text:               {demo['1_raw_text']}")
print(f"2. Lowercase:              {demo['2_lowercase']}")
print(f"3. Punctuation/Digits:     {demo['3_removed_punct_digits']}")
print(f"4. Raw Tokens ({demo['4_token_count']}):       {demo['4_tokens']}")
print(f"5. Filtered Tokens ({demo['5_filtered_token_count']}):  {demo['5_stopwords_removed']}")
print(f"6. Stemmed Tokens:         {demo['6_stemmed_tokens']}")
print(f"7. Lemmatized Tokens:      {demo['7_lemmatized_tokens']}")
print(f"\nDimensionality Reduction:   {demo['token_reduction_pct']}% of tokens eliminated as noise.")


## 5. TF-IDF Mathematical Feature Representation
We convert textual documents into high-dimensional numerical feature vectors using **TF-IDF**:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

Where:
$$\text{sublinear\_TF}(t, d) = 1 + \log(f_{t,d}) \quad (\text{for } f_{t,d} > 0)$$
$$\text{IDF}(t, D) = \log\left(\frac{1 + |D|}{1 + \text{DF}(t)}\right) + 1$$
$$\mathbf{v}(d) = \frac{\mathbf{w}(d)}{\|\mathbf{w}(d)\|_2}$$

**Preventing Data Leakage:**
In accordance with Information Retrieval best practices, `TfidfVectorizer` is placed inside an integrated Scikit-learn `Pipeline`.
The vocabulary and IDF values are fitted **strictly on the training split** ($X_{train}$), ensuring zero leakage of test set information.


## 6. Stratified Train-Test Split & Model Training
We partition the dataset into:
- **80% Training Set** (1,780 documents)
- **20% Testing Set** (445 documents)
- Stratified sampling guarantees identical class proportions across both splits.


In [ ]:
df_processed, label_encoder = load_and_preprocess_dataset('../dataset/bbc_news_clean.csv')
from src.model_training import train_and_cross_validate_models
X_train, X_test, y_train, y_test, pipelines, summary = train_and_cross_validate_models(df_processed, test_size=0.20, random_state=42)


## 7. Model Evaluation & Head-to-Head Comparative Battle
We evaluate all four models on the held-out 445 test documents.
Metrics computed include:
- **Accuracy**
- **Macro & Weighted Precision**
- **Macro & Weighted Recall**
- **Macro & Weighted F1-Score**
- **5-Fold Cross-Validation Accuracy**
- **Training Time & Prediction Latency**


In [ ]:
metrics_df, class_reports, confusion_matrices, labels = evaluate_all_models(
    pipelines, X_test, y_test, summary
)
print("\n=== OFFICIAL EXPERIMENTAL LEADERBOARD ===")
print(metrics_df[['Model', 'Accuracy (%)', 'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)', '5-Fold CV Mean', 'Training Time (s)', 'Latency (ms/doc)']].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
plot_df = metrics_df.melt(id_vars=['Model'], value_vars=['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)'], var_name='Metric', value_name='Score')
sns.barplot(data=plot_df, x='Model', y='Score', hue='Metric', ax=ax, palette='Blues_r')
ax.set_title("Performance Comparison Across Four Classifiers (BBC News Corpus)", weight='bold')
ax.set_ylim(0.70, 1.02)
plt.tight_layout()
plt.show()


## 8. Confusion Matrix & Per-Class Error Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11), dpi=150)
axes = axes.flatten()
for idx, (name, cm_list) in enumerate(confusion_matrices.items()):
    ax = axes[idx]
    cm = np.array(cm_list)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    annot = np.empty_like(cm).astype(str)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)"
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, xticklabels=labels, yticklabels=labels, cbar=False)
    ax.set_title(f"{name} Confusion Matrix", weight='bold')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True Ground Truth")
plt.tight_layout()
plt.show()


## 9. Empirical What-If Sensitivity Experiments
We conduct three empirical investigations:
1. **Experiment 1:** KNN Neighborhood size $k \in [1, 31]$ and Cosine vs. Euclidean distance.
2. **Experiment 2:** TF-IDF feature vocabulary capacity (`max_features`) from 250 to Unlimited.
3. **Experiment 3:** N-gram lexical span (Unigrams vs Bigrams vs Trigrams).


In [ ]:
df_knn = run_knn_sensitivity_experiment(X_train, X_test, y_train, y_test)
df_feat = run_tfidf_feature_capacity_experiment(X_train, X_test, y_train, y_test)
df_ngram = run_ngram_range_experiment(X_train, X_test, y_train, y_test)
plot_what_if_experiments(df_knn, df_feat, df_ngram)


## 10. Real-Time Inference & Explainability Demonstration

In [ ]:
predictor = NewsPredictor(model_name="best_model")
test_articles = [
    "Manchester United secured an emphatic victory in the Premier League derby match against Chelsea yesterday.",
    "Wall Street benchmark stock indices rallied after Federal Reserve signaled future interest rate reductions.",
    "Google DeepMind introduced advanced multi-modal artificial intelligence systems for automated software engineering.",
    "The Chancellor defended the fiscal economic policy in the House of Commons during Prime Minister Question Time.",
    "The musical drama won multiple Academy Awards at the prestigious Oscars ceremony held in Los Angeles."
]

for idx, article in enumerate(test_articles, 1):
    res = predictor.predict(article)
    print(f"Test {idx}: {article[:70]}...")
    print(f"  -> Predicted Category: {res['predicted_category'].upper()} (Confidence: {res['confidence']*100:.2f}%)")
    print(f"  -> Top Driving Keywords: {[w for w, s in res['top_keywords'][:4]]}\n")


## 11. Student Academic Defence & Conclusions

### Technical Defence Summary
1. **Support Vector Machine (Linear SVM) & Multinomial Naive Bayes tied for First Place (97.98% Test Accuracy):**
   - Linear SVM achieved the highest 5-Fold Cross Validation score (**98.48%**), demonstrating maximum-margin structural risk minimization in 5,000-dimensional sparse feature space.
   - Multinomial Naive Bayes achieved near-instantaneous training (**0.84s**) and inference (**0.16 ms/doc**), proving that its conditional independence assumption does not hinder class rank discrimination.
2. **K-Nearest Neighbours achieved 97.08% Test Accuracy with Cosine Distance:**
   - Cosine distance proved strictly superior to Euclidean distance by normalizing document vector lengths, eliminating article length bias.
3. **Decision Tree ranked Fourth (85.17% Test Accuracy):**
   - Axis-aligned orthogonal splits struggle in high-dimensional sparse textual feature spaces where class signals are dispersed across thousands of correlated words.

---
**Student Details:**
- **Name:** Sathish Kumar R
- **Register Number:** RA2411026050188
- **Institution:** SRM Institute of Science and Technology, Tiruchirappalli
- **Course:** 21CSE377T – Information Retrieval | CLA-2 Task 2
